<img src="imgs/unifor_logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br>
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos para Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

<p dir="ltr" style="text-align: left;">
    <strong>Links:</strong>
    <a href="https://www2.unifor.br/controle_pesquisa/pesquisarprofessor.do?actionParameter=prepareUpdate&amp;p_tp_ambiente=2&amp;p_tp_chamada=1&amp;p_tp_apresentacao=1&amp;cdPesquisador=767686193" target="_blank">Unifor</a> |
    <a href="http://lattes.cnpq.br/5738786447903616" target="_blank">Lattes</a> |
    <a href="https://unifor.br/web/pesquisa-inovacao/ncdia" target="_blank">NCDIA</a> |
    <a href="https://github.com/carubbi/" target="_blank">Github</a>
</p>

# **Teorema de Bayes**

## Em frente!

Neste notebook, vamos derivar três relações entre **conjunção** e **probabilidade condicional**:

* **Teorema 1:** Usando a conjunção para calcular uma probabilidade condicional.

* **Teorema 2:** Usando uma probabilidade condicional para calcular uma conjunção.

* **Teorema 3:** Usando `prob_cond(A, B)` para calcular `prob_cond(B, A)`.

O **Teorema 3** também é conhecido como **Teorema de Bayes**, que é a base da estatística Bayesiana.

Em algumas partes deste notebook será útil usar notação matemática para probabilidade, então vou introduzi-la agora:

* $P(A)$ é a probabilidade da proposição $A$.

* $P(A~\mathrm{and}~B)$ é a probabilidade da conjunção de $A$ e $B$, ou seja, a probabilidade de que ambos sejam verdadeiros.

* $P(A \mid B)$ é a probabilidade condicional de $A$ dado que $B$ é verdadeiro.  
  A barra vertical entre $A$ e $B$ é lida como **“dado que”**.

Com isso, estamos prontos para o **Teorema 1**.

In [1]:
# Importando as bibliotecas necessárias
import pandas as pd
import numpy as np

## Revisão

[No notebook anterior](aulas/notebooks/6_PRB.ipynb) defini **probabilidade**, **conjunção** e **probabilidade condicional**, e usei dados da *General Social Survey (GSS)* para calcular a probabilidade de várias proposições lógicas.

Para revisar, aqui está como carregamos o conjunto de dados:

In [2]:
# Carregando o dataset
gss = pd.read_csv('../../dataset/gss_bayes.csv')
gss.head()

,caseid,year,age,sex,polviews,partyid,indus10
0,1,1974,21.0,1,4.0,2.0,4970.0
1,2,1974,41.0,1,5.0,0.0,9160.0
2,5,1974,58.0,2,6.0,1.0,2670.0
3,6,1974,30.0,1,5.0,4.0,6870.0
4,7,1974,48.0,1,5.0,4.0,7860.0


In [3]:
# Renomeando as colunas para facilitar a manipulação
gss.rename(columns={
    'caseid': 'id',
    'year': 'ano',
    'age': 'idade',
    'sex': 'sexo',
    'polviews': 'visao_pol',
    'partyid': 'partido',
    'indus10': 'setor',
}, inplace=True)

gss.head()

,id,ano,idade,sexo,visao_pol,partido,setor
0,1,1974,21.0,1,4.0,2.0,4970.0
1,2,1974,41.0,1,5.0,0.0,9160.0
2,5,1974,58.0,2,6.0,1.0,2670.0
3,6,1974,30.0,1,5.0,4.0,6870.0
4,7,1974,48.0,1,5.0,4.0,7860.0


In [4]:
# Exibindo informações básicas sobre o dataset
gss.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49290 entries, 0 to 49289
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         49290 non-null  int64  
 1   ano        49290 non-null  int64  
 2   idade      49290 non-null  float64
 3   sexo       49290 non-null  int64  
 4   visao_pol  49290 non-null  float64
 5   partido    49290 non-null  float64
 6   setor      49290 non-null  float64
dtypes: float64(4), int64(3)
memory usage: 2.6 MB


As colunas são:

* `id`: identificador do respondente.
* `ano`: ano da entrevista.
* `idade`: idade do respondente na entrevista.
* `sexo`: sexo do respondente.
* `visao_pol`: posicionamento político em uma escala de liberal a conservador.
* `partido`: identificação partidária do respondente.
* `setor`: [código](https://www.census.gov/cgi-bin/sssd/naics/naicsrch?chart=2007) do setor de atividade econômica em que o respondente trabalha.

Vamos analisar essas variáveis em mais detalhes, começando por `setor`.

And here are the logical propositions we defined, represented using Boolean series.

In [5]:
setor_bancario = (gss['setor'] == 6870)

In [6]:
feminino = (gss['sexo'] == 2)

In [7]:
liberal = (gss['visao_pol'] < 4)

In [8]:
democrata = (gss['partido'] <= 1)

Defini a seguinte função, que usa `mean` para calcular a fração de valores `True` em uma série booleana.

In [9]:
def prob(A):
    """Calcula a probabilidade de uma proposição A.
    A: série booleana
    retorna: probabilidade
    """
    assert isinstance(A, pd.Series)
    assert A.dtype == 'bool'

    return A.mean()

So podemos compute the probabilidade of a proposition like this:

In [10]:
prob(feminino)

np.float64(0.5378575776019476)

Então usamos o operador `&` para calcular a **probabilidade de uma conjunção**, assim:

In [11]:
prob(feminino & setor_bancario)

np.float64(0.011381618989653074)

Em seguida, defini a seguinte função, que usa o operador de colchetes para calcular a **probabilidade condicional**:

In [12]:
def prob_cond(A, B):
    """Calcula a probabilidade condicional de A dado B.

    A: série booleana
    B: série booleana

    retorna: probabilidade
    """
    return prob(A[B])

Mostramos que a conjunção é comutativa, portanto `prob(A & B)` é igual a `prob(B & A)`, para quaisquer proposições lógicas `A` e `B`.

Por exemplo:

In [13]:
prob(liberal & democrata)

np.float64(0.1425238385067965)

In [14]:
prob(democrata & liberal)

np.float64(0.1425238385067965)

Mas a **probabilidade condicional NÃO é comutativa**, portanto `prob_cond(A, B)` geralmente não é o mesmo que `prob_cond(B, A)`.

Por exemplo, aqui está a probabilidade de que um entrevistado seja mulher, dado que ele seja bancário.

In [15]:
prob_cond(feminino, setor_bancario)

np.float64(0.7706043956043956)

E aqui está a probabilidade de que um entrevistado seja bancário, dado que seja mulher.

In [16]:
prob_cond(setor_bancario, feminino)

np.float64(0.02116102749801969)

Nem de perto.

## Mais proposições

Para variar nossos **exemplos**, vamos definir algumas novas proposições.

Aqui está a probabilidade de que um entrevistado aleatório seja homem.

In [17]:
masculino = (gss['sexo']==1)
prob(masculino)

np.float64(0.46214242239805237)

O código da indústria para **“Construção”** é `770`.  
Vamos chamar alguém desta área de **“construtor”**.

In [18]:
construtor = (gss['setor'] == 770)
prob(construtor)

np.float64(0.05978900385473727)

E vamos definir proposições para **conservadores** e **republicanos**.

In [19]:
conservador = (gss['visao_pol'] > 4)
prob(conservador)

np.float64(0.3419354838709677)

In [20]:
republicano = (gss['partido'].isin([5,6]))
prob(republicano)

np.float64(0.2610062893081761)

Por fim, usarei `idade` para definir as proposições `jovem` e `idoso`.

In [21]:
jovem = (gss['idade'] < 30)
prob(jovem)

np.float64(0.19435991073240008)

In [22]:
idoso = (gss['idade'] >= 65)
prob(idoso)

np.float64(0.17328058429701765)

Para esses limites, escolhi números arredondados próximos aos percentis 20 e 80.  
Dependendo da sua idade, você pode ou não concordar com essas definições de *“jovem”* e *“idoso”*.

**Exercício:** Há uma [citação famosa](https://quoteinvestigator.com/2014/02/24/heart-head/) sobre jovens, pessoas idosas, liberais e conservadores que diz algo como:

> Se você não é liberal aos 25 anos, não tem coração.  
> Se você não é conservador aos 35 anos, não tem cérebro.

Concordando ou não com essa proposição, ela sugere algumas probabilidades que podemos calcular como um exercício de revisão.  
Use `prob` e `prob_cond` para calcular essas probabilidades.

* Qual é a **probabilidade** de que um entrevistado escolhido aleatoriamente seja um **jovem liberal**?

* Qual é a **probabilidade** de que uma **pessoa jovem** seja **liberal**?

* Que **fração** dos entrevistados são **idosos conservadores**?

* Que **fração** dos **conservadores** são **idosos**?

Para cada enunciado, pense se ele expressa uma **conjunção**, uma **probabilidade condicional** ou ambos.

E, nas probabilidades condicionais, **cuidado com a ordem**!

In [23]:
# Solução


In [24]:
# Solução


In [25]:
# Solução


In [26]:
# Solução


Se a sua última resposta for maior que **30%**, você colocou ao contrário!

## Em frente!

Neste notebook, vamos derivar três relações entre **conjunção** e **probabilidade condicional**:

* **Teorema 1:** Usando a conjunção para calcular uma probabilidade condicional.

* **Teorema 2:** Usando uma probabilidade condicional para calcular uma conjunção.

* **Teorema 3:** Usando `prob_cond(A, B)` para calcular `prob_cond(B, A)`.

O **Teorema 3** também é conhecido como **Teorema de Bayes**, que é a base da estatística Bayesiana.

Em algumas partes deste notebook será útil usar notação matemática para probabilidade, então vou introduzi-la agora:

* $P(A)$ é a probabilidade da proposição $A$.

* $P(A~\mathrm{and}~B)$ é a probabilidade da conjunção de $A$ e $B$, ou seja, a probabilidade de que ambos sejam verdadeiros.

* $P(A \mid B)$ é a probabilidade condicional de $A$ dado que $B$ é verdadeiro.  
  A barra vertical entre $A$ e $B$ é lida como **“dado que”**.

Com isso, estamos prontos para o **Teorema 1**.

## Teorema 1

Que fração dos **construtores** são homens?  
Já vimos uma maneira de calcular a resposta:

1. Usar o operador de colchetes para selecionar os construtores, depois  

2. Usar `mean` para calcular a fração de construtores que são homens.

Podemos escrever esses passos assim:

In [27]:
masculino[construtor].mean()

np.float64(0.8920936545639634)

Ou podemos usar a função `prob_cond`, que faz a mesma coisa:

In [28]:
prob_cond(masculino, construtor)

np.float64(0.8920936545639634)

Mas há outra maneira: para calcular a fração de construtores que são homens, podemos calcular a razão entre duas probabilidades:

1. A fração dos entrevistados que são **homens construtores**, e  

2. A fração dos entrevistados que são **construtores**.

Aqui está como isso fica:

In [29]:
prob(masculino & construtor) / prob(construtor)

np.float64(0.8920936545639634)

O resultado é o mesmo.

Este **exemplo** demonstra uma regra geral que relaciona **probabilidade condicional** e **conjunção**.  
Em notação matemática, isso fica assim:

$$
P(A \mid B) = \frac{P(A~\mathrm{and}~B)}{P(B)}
$$

E esse é o **Teorema 1**.

Neste exemplo:

`prob_cond(masculino, construtor) = prob(masculino & construtor) / prob(construtor)`

**Exercício:** Que fração dos **conservadores** são **republicanos**? Calcule a resposta de duas maneiras:

* Usando `prob_cond` (que por baixo dos panos utiliza o operador de colchetes), e
* Usando o **Teorema 1**.

Confirme que você obtém o mesmo resultado.

**Observação:** Devido à aritmética de ponto flutuante, os resultados podem não ser exatamente idênticos, mas quase todos os dígit

In [30]:
# Solução


In [31]:
# Solução


## Prova?

Eu não cheguei a **provar** o Teorema 1; em essência, ele é uma afirmação sobre o que **probabilidade condicional** significa.

Por exemplo, considere este diagrama de Venn:

<img width="200" src="https://github.com/AllenDowney/BiteSizeBayes/raw/master/theorem1_venn_diagram.png">

O círculo azul representa os respondentes **homens**.  
O círculo vermelho representa os **construtores**.  
A interseção representa os **homens construtores**.

Para calcular a fração de construtores que são homens, podemos calcular a razão entre a **interseção**, que é `prob(masculino & constrtutor)`, e o círculo **vermelho**, que é `prob(construtor)`.

**Exercício:** Para praticar, calcule a fração de **banqueiros** que são **idosos** de duas formas: usando `prob_cond` e usando o **Teorema 1**.

In [32]:
# Solução


In [33]:
# Solução


## Teorema 2

Aqui está o Teorema 1 novamente:

$$
P(A \mid B) = \frac{P(A~\mathrm{and}~B)}{P(B)}
$$

Se multiplicarmos ambos os lados por $P(B)$, obtemos o **Teorema 2**:

$$
P(A~\mathrm{and}~B) = P(B)\, P(A \mid B)
$$

Essa fórmula sugere uma segunda maneira de calcular uma **conjunção**: em vez de usar o operador `&`, podemos calcular o **produto** de duas probabilidades.

Vamos ver se isso funciona para `conservador` e `republicano`.  
Aqui está o resultado usando `&`:

In [34]:
prob(conservador & republicano)

np.float64(0.15396632176912153)

E aqui está o resultado usando o **Teorema 2**:

In [35]:
prob(republicano) * prob_cond(conservador, republicano)

np.float64(0.1539663217691215)

Devido a erros de ponto flutuante, eles podem não ser idênticos, mas quase todos os dígitos são os mesmos.

**Exercício:** Verifique o **Teorema 2** mais uma vez calculando a fração de entrevistados que são **idosos liberais** de duas formas:

* Usando o operador `&`, e
* Usando o **Teorema 2**.

Os resultados devem ser iguais ou, pelo menos, **muito próximos**.

In [36]:
# Solutção


In [37]:
# Solução


## Conjunção é comutativa

Já estabelecemos que a **conjunção é comutativa**. Em notação matemática, isso significa:

$$
P(A~\mathrm{and}~B) = P(B~\mathrm{and}~A)
$$

Se aplicarmos o **Teorema 2** aos dois lados, temos:

$$
P(B)\, P(A \mid B) = P(A)\, P(B \mid A)
$$

Uma forma de interpretar isso é: se você quer verificar $A$ e $B$, pode fazê-lo em **qualquer ordem**:

1. Você pode verificar **$B$ primeiro**, depois **$A$ condicionado a $B$**, ou  
2. Você pode verificar **$A$ primeiro**, depois **$B$ condicionado a $A$**.

Para experimentar, vou calcular a fração de **jovens construtores** das duas maneiras:

In [38]:
prob(jovem) * prob_cond(construtor, jovem)

np.float64(0.012314871170622844)

In [39]:
prob(construtor) * prob_cond(jovem, construtor)

np.float64(0.012314871170622844)

Mesma coisa!

**Exercício:** Calcule a **probabilidade** de ser um **homem banqueiro** das duas maneiras e veja se você obtém o mesmo resultado.

In [40]:
# Solutção


In [41]:
# Solução


## Teorema 3

Na seção anterior estabelecemos que

$$
P(B)\, P(A \mid B) = P(A)\, P(B \mid A)
$$

Se dividirmos ambos os lados por $P(B)$, obtemos o **Teorema 3**:

$$
P(A \mid B) = \frac{P(A)\, P(B \mid A)}{P(B)}
$$

E isso, meus amigos, é o **Teorema de Bayes**.

Para ver como ele funciona, vamos tentar mais uma combinação das nossas proposições.  
Vamos calcular a fração de **construtores que são liberais**, primeiro usando `prob_cond`:

In [42]:
prob_cond(liberal, construtor)

np.float64(0.24431625381744146)

Agora usando o Teorema de Bayes:

In [43]:
prob(liberal) * prob_cond(construtor, liberal) / prob(construtor)

np.float64(0.24431625381744151)

Mesma coisa!

**Exercício:** Tente você mesmo! Calcule a fração de **jovens** que são **republicanos** de duas maneiras: usando `prob_cond` e usando o **Teorema de Bayes**. Veja se você obtém o mesmo resultado.

In [44]:
prob_cond(republicano, jovem)

np.float64(0.23319415448851774)

In [45]:
prob(republicano) * prob_cond(jovem, republicano) / prob(jovem)

np.float64(0.2331941544885177)

## Resumo

Aqui está o que temos até agora:

**Teorema 1** nos dá uma nova forma de calcular uma **probabilidade condicional** usando uma conjunção:

$$
P(A \mid B) = \frac{P(A~\mathrm{and}~B)}{P(B)}
$$

**Teorema 2** nos dá uma nova forma de calcular uma **conjunção** usando uma probabilidade condicional:

$$
P(A~\mathrm{and}~B) = P(B)\, P(A \mid B)
$$

**Teorema 3**, também conhecido como **Teorema de Bayes**, nos dá uma forma de passar de $P(A \mid B)$ para $P(B \mid A)$, ou vice-versa:

$$
P(A \mid B) = \frac{P(A)\, P(B \mid A)}{P(B)}
$$

Mas, neste ponto, você pode perguntar: **“E daí?”**  
Se temos todos os dados, podemos calcular qualquer probabilidade que quisermos, qualquer conjunção ou qualquer probabilidade condicional, simplesmente contando.  
Por que precisamos dessas fórmulas?

E você está certo, *se* tivermos todos os dados.  
Mas muitas vezes **não temos**, e nesse caso, essas fórmulas podem ser bastante úteis — especialmente o **Teorema de Bayes**.